In [2]:
import subprocess
import sys

cmd = [sys.executable, "-m", "pip", "install", "optuna"]
print("Installing Optuna with:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Optuna installed.")

Installing Optuna with: /usr/bin/python3 -m pip install optuna
Optuna installed.


In [3]:
try:
    import optuna

    print("optuna_available:", True, "version:", optuna.__version__)
except Exception as e:
    print("optuna_available:", False)
    print(type(e).__name__, str(e)[:200])

optuna_available: True version: 4.8.0


In [ ]:
import re
from pathlib import Path

train_file = globals().get(
    "TRAIN_FILE",
    Path("/content/v2e/research/v2e_imu/train.py"),
)

text = train_file.read_text(encoding="utf-8")
for k in [
    "TOTAL_BATCH_SIZE",
    "DEVICE_BATCH_SIZE",
    "LEARNING_RATE",
    "WEIGHT_DECAY",
    "WARMUP_RATIO",
    "WARMDOWN_RATIO",
    "FINAL_LR_FRAC",
    "BASE_CHANNELS",
    "IMU_HIDDEN_DIM",
    "MODEL_TYPE",
]:
    m = re.search(rf"^{k}\s*=\s*(.+)$", text, flags=re.MULTILINE)
    print(k, "=", m.group(1) if m else "<missing>")

TOTAL_BATCH_SIZE = 4
DEVICE_BATCH_SIZE = 4
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.1
WARMDOWN_RATIO = 0.3
FINAL_LR_FRAC = 0.01
BASE_CHANNELS = 32
IMU_HIDDEN_DIM = 128
MODEL_TYPE = "unet"


In [13]:
from pathlib import Path

base = Path("/content/v2e/research/v2e_imu/data/fpv")
print("base_exists:", base.exists())

if base.exists():
    seq_dirs = sorted([p for p in base.iterdir() if p.is_dir()])
    print("sequence_dirs:", [p.name for p in seq_dirs])

    txt_files = list(base.rglob("*.txt"))
    bag_files = list(base.rglob("*.bag"))
    zip_files = list(base.rglob("*.zip"))

    print("txt_count:", len(txt_files))
    print("bag_count:", len(bag_files))
    print("zip_count (should usually be 0 after extraction):", len(zip_files))

    # Basic robustness checks: non-empty key sensor text files expected in a sequence folder
    expected_any = {"events.txt", "imu.txt", "images.txt"}
    found_names = {p.name for p in txt_files}
    print("contains_expected_sensor_files:", bool(expected_any & found_names))

    # Show top candidates with sizes
    for p in sorted(txt_files)[:15]:
        print("txt_file:", p, "bytes=", p.stat().st_size)
else:
    print("dataset missing")

base_exists: True
sequence_dirs: ['indoor_forward_3']
txt_count: 4
bag_count: 0
zip_count (should usually be 0 after extraction): 0
contains_expected_sensor_files: True
txt_file: /content/v2e/research/v2e_imu/data/fpv/indoor_forward_3/events.txt bytes= 1462109130
txt_file: /content/v2e/research/v2e_imu/data/fpv/indoor_forward_3/groundtruth.txt bytes= 3523904
txt_file: /content/v2e/research/v2e_imu/data/fpv/indoor_forward_3/images.txt bytes= 104447
txt_file: /content/v2e/research/v2e_imu/data/fpv/indoor_forward_3/imu.txt bytes= 11074623


In [20]:
import datetime as dt
import os
import re
import subprocess
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/vlordier/v2e.git"
DEFAULT_BRANCH = "upgrades"
DEFAULT_FPV_SEQUENCE = os.getenv("FPV_SEQUENCE", "indoor_forward_3").strip() or "indoor_forward_3"


def run_cmd(
    cmd: str,
    cwd: Path | None = None,
    check: bool = True,
    timeout: int | None = None,
) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=True,
        check=check,
        text=True,
        capture_output=True,
        timeout=timeout,
    )


def ensure_repo(url: str, branch: str, dst: Path) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        print(f"Cloning {url} (branch={branch}) to {dst} ...")
        run_cmd(f"git clone -b {branch} {url} {dst}", cwd=dst.parent, check=True)
    elif not (dst / ".git").exists():
        raise RuntimeError(f"{dst} exists but is not a git repo")

    # Enforce remote URL and target branch for reproducibility.
    run_cmd(f"git remote set-url origin {url}", cwd=dst, check=True)
    run_cmd("git fetch origin --prune", cwd=dst, check=True)
    run_cmd(f"git checkout {branch}", cwd=dst, check=True)
    run_cmd(f"git reset --hard origin/{branch}", cwd=dst, check=True)
    return dst


def find_or_clone_repo() -> Path:
    repo_url = os.getenv("V2E_REPO_URL", DEFAULT_REPO_URL).strip()
    branch = os.getenv("V2E_BRANCH", DEFAULT_BRANCH).strip() or DEFAULT_BRANCH
    target = Path("/content/v2e")
    return ensure_repo(repo_url, branch, target)


def find_research_dir(root: Path) -> Path:
    preferred = root / "research" / "v2e_imu"
    if (
        (preferred / "program.md").exists()
        and (preferred / "train.py").exists()
        and (preferred / "prepare_data.py").exists()
    ):
        return preferred

    for p in root.rglob("program.md"):
        parent = p.parent
        if (parent / "train.py").exists() and (parent / "prepare_data.py").exists():
            return parent

    raise FileNotFoundError(
        "Missing research/v2e_imu experiment files in remote kernel clone. "
        f"Checked repo: {root}. Confirm branch {DEFAULT_BRANCH} contains "
        "research/v2e_imu/program.md, train.py, and prepare_data.py."
    )


ROOT = find_or_clone_repo()
RESEARCH_DIR = find_research_dir(ROOT)
TRAIN_FILE = RESEARCH_DIR / "train.py"
PREPARE_FILE = RESEARCH_DIR / "prepare_data.py"
PROGRAM_FILE = RESEARCH_DIR / "program.md"
RESULTS_TSV = RESEARCH_DIR / "results.tsv"
RUN_LOG = RESEARCH_DIR / "run.log"


def sh(cmd: str, cwd: Path | None = None, timeout: int | None = None) -> str:
    result = run_cmd(cmd, cwd=cwd or ROOT, check=True, timeout=timeout)
    return result.stdout.strip()


def sh_allow_fail(
    cmd: str, cwd: Path | None = None, timeout: int | None = None
) -> tuple[int, str, str]:
    result = run_cmd(cmd, cwd=cwd or ROOT, check=False, timeout=timeout)
    return result.returncode, result.stdout.strip(), result.stderr.strip()


def parse_run_metrics(log_text: str) -> tuple[float | None, float | None]:
    val_ap_match = re.search(r"^val_ap:\s*([0-9]*\.?[0-9]+)", log_text, flags=re.MULTILINE)
    vram_match = re.search(r"^peak_vram_mb:\s*([0-9]*\.?[0-9]+)", log_text, flags=re.MULTILINE)
    val_ap = float(val_ap_match.group(1)) if val_ap_match else None
    peak_vram_mb = float(vram_match.group(1)) if vram_match else None
    return val_ap, peak_vram_mb


def ensure_results_tsv() -> None:
    if not RESULTS_TSV.exists() or RESULTS_TSV.read_text(encoding="utf-8").strip() == "":
        RESULTS_TSV.write_text(
            "commit\tval_ap\tpeak_memory_gb\tstatus\tdescription\n", encoding="utf-8"
        )


def append_result(
    commit: str, val_ap: float, peak_vram_mb: float, status: str, description: str
) -> None:
    peak_gb = 0.0 if peak_vram_mb is None else round(peak_vram_mb / 1024.0, 1)
    line = f"{commit}\t{val_ap:.6f}\t{peak_gb:.1f}\t{status}\t{description}\n"
    with RESULTS_TSV.open("a", encoding="utf-8") as f:
        f.write(line)


def current_branch() -> str:
    return sh("git rev-parse --abbrev-ref HEAD", cwd=ROOT)


def short_commit() -> str:
    return sh("git rev-parse --short HEAD", cwd=ROOT)


def ensure_research_branch() -> None:
    today_tag = dt.datetime.now().strftime("%b%d").lower()
    desired = f"research/{today_tag}"
    branch = current_branch()
    if branch.startswith("research/"):
        print(f"Using existing research branch: {branch}")
        return

    rc, _, _ = sh_allow_fail(f"git rev-parse --verify {desired}", cwd=ROOT)
    if rc == 0:
        sh(f"git checkout {desired}", cwd=ROOT)
        print(f"Checked out existing branch {desired}")
    else:
        sh(f"git checkout -b {desired}", cwd=ROOT)
        print(f"Created and switched to branch {desired}")


def ensure_real_fpv_data() -> None:
    data_dir = RESEARCH_DIR / "data" / "fpv"
    has_data = data_dir.exists() and any(data_dir.rglob("*.txt"))
    if has_data:
        return

    download_script = RESEARCH_DIR / "download_fpv.py"
    if not download_script.exists():
        raise FileNotFoundError(f"Missing downloader script: {download_script}")

    print(f"FPV data missing at {data_dir}. Downloading real sequence: {DEFAULT_FPV_SEQUENCE} ...")
    cmd = f"python {download_script.name} --sequence {DEFAULT_FPV_SEQUENCE} --output data/fpv/"
    proc = run_cmd(cmd, cwd=RESEARCH_DIR, check=False)
    if proc.returncode != 0:
        print(proc.stdout[-2000:])
        print(proc.stderr[-2000:])
        raise RuntimeError(
            "FPV download failed. Check network access and sequence name, "
            f"or set FPV_SEQUENCE. Attempted sequence: {DEFAULT_FPV_SEQUENCE}"
        )

    has_data_after = data_dir.exists() and any(data_dir.rglob("*.txt"))
    if not has_data_after:
        raise RuntimeError(f"FPV download completed but no data files found under {data_dir}")


def verify_setup() -> None:
    missing = [p for p in (TRAIN_FILE, PREPARE_FILE, PROGRAM_FILE) if not p.exists()]
    if missing:
        missing_str = ", ".join(str(p) for p in missing)
        raise FileNotFoundError(f"Missing required files: {missing_str}")

    ensure_real_fpv_data()


def run_train_with_timeout() -> tuple[bool, str]:
    cmd = f"python train.py > {RUN_LOG.name} 2>&1"
    try:
        sh(cmd, cwd=RESEARCH_DIR, timeout=20 * 60)
        return True, RUN_LOG.read_text(encoding="utf-8", errors="replace")
    except subprocess.TimeoutExpired:
        sh_allow_fail("pkill -f 'python train.py'", cwd=RESEARCH_DIR)
        log_text = RUN_LOG.read_text(encoding="utf-8", errors="replace") if RUN_LOG.exists() else ""
        return False, log_text


def install_deps_if_needed() -> None:
    probe = run_cmd(
        'python -c "import torch, torchvision, numpy, yaml"',
        cwd=ROOT,
        check=False,
    )
    if probe.returncode == 0:
        return

    print("Installing Python dependencies for remote kernel...")
    req = ROOT / "requirements.txt"
    if req.exists():
        sh("python -m pip install -r requirements.txt", cwd=ROOT)
    else:
        sh("python -m pip install -e .", cwd=ROOT)


def run_baseline_if_needed() -> None:
    ensure_results_tsv()
    existing = RESULTS_TSV.read_text(encoding="utf-8")
    if "\tbaseline" in existing:
        print("Baseline already logged; skipping baseline rerun.")
        return

    print("Running baseline (15-minute train budget)...")
    ok, log_text = run_train_with_timeout()
    commit = short_commit()
    val_ap, peak_vram_mb = parse_run_metrics(log_text)

    if ok and val_ap is not None and peak_vram_mb is not None:
        append_result(commit, val_ap, peak_vram_mb, "keep", "baseline")
        print(f"Baseline complete: val_ap={val_ap:.6f}, peak_vram_mb={peak_vram_mb:.1f}")
    else:
        append_result(commit, 0.0, 0.0, "crash", "baseline crashed or timed out")
        tail = "\n".join(log_text.splitlines()[-50:])
        print("Baseline crashed or timed out. Last log lines:\n")
        print(tail)


os.chdir(ROOT)
install_deps_if_needed()
verify_setup()
ensure_research_branch()
run_baseline_if_needed()

print("\nAutoresearch notebook setup complete.")
print(f"Repo root: {ROOT}")
print(f"Research dir: {RESEARCH_DIR}")
print(f"Current branch: {current_branch()}")
print(f"Current commit: {short_commit()}")
print(f"Results TSV: {RESULTS_TSV}")
print(f"FPV sequence target: {DEFAULT_FPV_SEQUENCE}")
print("Edit train.py in RESEARCH_DIR between runs to test new ideas.")

Checked out existing branch research/apr03
Baseline already logged; skipping baseline rerun.

Autoresearch notebook setup complete.
Repo root: /content/v2e
Research dir: /content/v2e/research/v2e_imu
Current branch: research/apr03
Current commit: 76e60c6
Results TSV: /content/v2e/research/v2e_imu/results.tsv
FPV sequence target: indoor_forward_3
Edit train.py in RESEARCH_DIR between runs to test new ideas.


In [34]:
# Optuna-guided small autoresearch batch (val_ap objective)
import csv
import json
import re
from pathlib import Path

import optuna

N_TRIALS = 3
# Store DB in /tmp — never committed to git, so it can't become read-only after pull.
OPTUNA_DB = Path("/tmp/optuna_v2e_small.db")


def load_results_rows(tsv_path: Path) -> list[dict[str, str]]:
    if not tsv_path.exists():
        return []
    with tsv_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        return [r for r in reader]


def best_keep_val_ap(tsv_path: Path) -> float:
    best = 0.0
    for row in load_results_rows(tsv_path):
        try:
            if row.get("status") == "keep":
                best = max(best, float(row.get("val_ap", "0") or 0.0))
        except ValueError:
            pass
    return best


def ensure_git_identity() -> None:
    rc_n, out_n, _ = sh_allow_fail("git config user.name", cwd=ROOT)
    rc_e, out_e, _ = sh_allow_fail("git config user.email", cwd=ROOT)
    if rc_n != 0 or not out_n.strip():
        sh('git config user.name "autoresearch-bot"', cwd=ROOT)
    if rc_e != 0 or not out_e.strip():
        sh('git config user.email "autoresearch-bot@example.com"', cwd=ROOT)


def set_constant(path: Path, name: str, value_expr: str) -> None:
    text = path.read_text(encoding="utf-8")
    pattern = rf"^{name}\s*=\s*.*$"
    replacement = f"{name} = {value_expr}"
    new_text, n = re.subn(pattern, replacement, text, count=1, flags=re.MULTILINE)
    if n != 1:
        raise RuntimeError(f"Could not set constant {name} in {path}")
    path.write_text(new_text, encoding="utf-8")


def apply_trial_params(params: dict[str, str]) -> None:
    # Keep proven best throughput setup fixed.
    set_constant(TRAIN_FILE, "TOTAL_BATCH_SIZE", "4")
    set_constant(TRAIN_FILE, "DEVICE_BATCH_SIZE", "4")
    set_constant(TRAIN_FILE, "MODEL_TYPE", '"unet"')
    set_constant(TRAIN_FILE, "BASE_CHANNELS", "32")
    set_constant(TRAIN_FILE, "IMU_HIDDEN_DIM", "128")

    for k, v in params.items():
        set_constant(TRAIN_FILE, k, v)


def run_one_trial(trial: optuna.Trial) -> float:
    params = {
        "LEARNING_RATE": trial.suggest_categorical("LEARNING_RATE", ["2e-3", "1.5e-3", "1e-3"]),
        "WEIGHT_DECAY": trial.suggest_categorical("WEIGHT_DECAY", ["0.0", "1e-5", "1e-4"]),
        "WARMUP_RATIO": trial.suggest_categorical("WARMUP_RATIO", ["0.05", "0.1"]),
        "WARMDOWN_RATIO": trial.suggest_categorical("WARMDOWN_RATIO", ["0.3", "0.5"]),
        "FINAL_LR_FRAC": trial.suggest_categorical("FINAL_LR_FRAC", ["0.01", "0.02", "0.05"]),
    }

    best_before = best_keep_val_ap(RESULTS_TSV)
    print(f"\nTrial {trial.number}: params={params}")
    print(f"Best keep val_ap before trial: {best_before:.6f}")

    apply_trial_params(params)

    rel_train = str(TRAIN_FILE.relative_to(ROOT))
    sh(f"git add {rel_train}", cwd=ROOT)

    rc_diff, _, _ = sh_allow_fail("git diff --cached --quiet", cwd=ROOT)
    if rc_diff == 0:
        print("No net change from current config; skipping trial run.")
        return best_before

    desc = (
        f"optuna lr={params['LEARNING_RATE']} wd={params['WEIGHT_DECAY']} "
        f"warmup={params['WARMUP_RATIO']} warmdown={params['WARMDOWN_RATIO']} "
        f"finalfrac={params['FINAL_LR_FRAC']}"
    )

    rc_commit, out_commit, err_commit = sh_allow_fail(
        f'git commit -m "experiment: {desc}"',
        cwd=ROOT,
    )
    if rc_commit != 0:
        raise RuntimeError(f"git commit failed\nSTDOUT:\n{out_commit}\nSTDERR:\n{err_commit}")

    exp_commit = short_commit()
    print(f"Running commit {exp_commit}")

    ok, _ = run_train_with_timeout()
    log_text = RUN_LOG.read_text(encoding="utf-8", errors="replace") if RUN_LOG.exists() else ""
    val_ap, peak_vram_mb = parse_run_metrics(log_text)

    if (not ok) or val_ap is None or peak_vram_mb is None:
        append_result(exp_commit, 0.0, 0.0, "crash", desc)
        print("Trial crashed/timed out; reverting.")
        sh("git reset --hard HEAD~1", cwd=ROOT)
        return 0.0

    improved = val_ap > best_before
    status = "keep" if improved else "discard"
    append_result(exp_commit, val_ap, peak_vram_mb, status, desc)

    if improved:
        sh(f"git add {RESULTS_TSV.relative_to(ROOT)}", cwd=ROOT)
        sh("git commit --amend --no-edit", cwd=ROOT)
        print(f"KEEP val_ap={val_ap:.6f} peak_vram_mb={peak_vram_mb:.1f}")
    else:
        print(f"DISCARD val_ap={val_ap:.6f} (best={best_before:.6f}); reverting")
        sh("git reset --hard HEAD~1", cwd=ROOT)

    return val_ap


def run_optuna_small_batch(n_trials: int = N_TRIALS) -> optuna.study.Study:
    os.chdir(ROOT)
    ensure_git_identity()

    storage = f"sqlite:///{OPTUNA_DB}"
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(
        study_name="v2e_val_ap_small",
        direction="maximize",
        sampler=sampler,
        storage=storage,
        load_if_exists=True,
    )

    study.optimize(run_one_trial, n_trials=n_trials)

    summary = {
        "best_value": study.best_value if len(study.trials) else None,
        "best_params": study.best_params if len(study.trials) else {},
        "completed_trials": len(
            [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        ),
        "total_trials": len(study.trials),
    }
    (RESEARCH_DIR / "optuna_small_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )

    print("\nOptuna small batch complete:")
    print(json.dumps(summary, indent=2))
    print("\nresults.tsv tail:")
    print("\n".join(RESULTS_TSV.read_text(encoding="utf-8").splitlines()[-10:]))

    return study


study = run_optuna_small_batch(n_trials=N_TRIALS)

[I 2026-04-03 12:43:14,160] Using an existing study with name 'v2e_val_ap_small' instead of creating a new one.



Trial 3: params={'LEARNING_RATE': '1.5e-3', 'WEIGHT_DECAY': '0.0', 'WARMUP_RATIO': '0.1', 'WARMDOWN_RATIO': '0.5', 'FINAL_LR_FRAC': '0.02'}
Best keep val_ap before trial: 0.586500
Running commit 6beec81


[I 2026-04-03 13:01:12,398] Trial 3 finished with value: 0.5054 and parameters: {'LEARNING_RATE': '1.5e-3', 'WEIGHT_DECAY': '0.0', 'WARMUP_RATIO': '0.1', 'WARMDOWN_RATIO': '0.5', 'FINAL_LR_FRAC': '0.02'}. Best is trial 2 with value: 0.5126.


DISCARD val_ap=0.505400 (best=0.586500); reverting

Trial 4: params={'LEARNING_RATE': '2e-3', 'WEIGHT_DECAY': '1e-5', 'WARMUP_RATIO': '0.1', 'WARMDOWN_RATIO': '0.5', 'FINAL_LR_FRAC': '0.05'}
Best keep val_ap before trial: 0.586500
Running commit fdf50f8


[I 2026-04-03 13:18:50,867] Trial 4 finished with value: 0.5072 and parameters: {'LEARNING_RATE': '2e-3', 'WEIGHT_DECAY': '1e-5', 'WARMUP_RATIO': '0.1', 'WARMDOWN_RATIO': '0.5', 'FINAL_LR_FRAC': '0.05'}. Best is trial 2 with value: 0.5126.


DISCARD val_ap=0.507200 (best=0.586500); reverting

Trial 5: params={'LEARNING_RATE': '1e-3', 'WEIGHT_DECAY': '1e-5', 'WARMUP_RATIO': '0.1', 'WARMDOWN_RATIO': '0.3', 'FINAL_LR_FRAC': '0.05'}
Best keep val_ap before trial: 0.586500
Running commit b574afe


[I 2026-04-03 13:36:33,980] Trial 5 finished with value: 0.4981 and parameters: {'LEARNING_RATE': '1e-3', 'WEIGHT_DECAY': '1e-5', 'WARMUP_RATIO': '0.1', 'WARMDOWN_RATIO': '0.3', 'FINAL_LR_FRAC': '0.05'}. Best is trial 2 with value: 0.5126.


DISCARD val_ap=0.498100 (best=0.586500); reverting

Optuna small batch complete:
{
  "best_value": 0.5126,
  "best_params": {
    "LEARNING_RATE": "1e-3",
    "WEIGHT_DECAY": "1e-5",
    "WARMUP_RATIO": "0.1",
    "WARMDOWN_RATIO": "0.3",
    "FINAL_LR_FRAC": "0.05"
  },
  "completed_trials": 6,
  "total_trials": 6
}

results.tsv tail:
9f72407	0.5718	0.0	discard	WEIGHT_DECAY=1e-4 (marginal, precision dropped)
afd2068	0.5651	0.0	discard	focal gamma=3.0 (stronger suppression, no precision improvement)
af37f9d	0.5720	0.0	discard	WARMUP=5% WARMDOWN=50% (closest so far, precision 0.497 vs 0.488)
87e7a3a	0.5757	0.0	keep	TOTAL_BATCH_SIZE=16 (2x gradient updates, +0.0023 over baseline)
4e07ef3	0.5742	0.0	discard	batch=16 + WARMUP=5% WARMDOWN=50% (didn't stack, worse than batch=16 alone)
f25d0dc	0.5776	0.0	keep	TOTAL_BATCH_SIZE=8 (4x gradient updates, +0.0019 over batch=16)
c5e423b	0.5865	0.0	keep	TOTAL_BATCH_SIZE=4 (no accum, max updates, +0.0089 over batch=8)
baf3a14	0.5842	0.0	discard	TOTAL_

In [17]:
import base64
import getpass
import os
import subprocess
from pathlib import Path

repo = Path(globals().get("ROOT", Path("/content/v2e")))
branch = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    cwd=repo,
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()


def get_github_token() -> str:
    token = os.getenv("GITHUB_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata  # type: ignore

        token = userdata.get("GITHUB_TOKEN") or ""
        token = token.strip()
        if token:
            return token
    except Exception:
        pass
    return getpass.getpass("GitHub token (repo push scope): ").strip()


def git(args: list[str], check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        ["git", *args],
        cwd=repo,
        check=check,
        text=True,
        capture_output=True,
    )


token = get_github_token()
if not token:
    raise RuntimeError("Missing GitHub token")

basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode("ascii")
extraheader_key = "http.https://github.com/.extraheader"
extraheader_value = f"AUTHORIZATION: basic {basic_auth}"

git(["config", "--local", extraheader_key, extraheader_value])
try:
    print("repo:", repo)
    print("branch:", branch)
    print(git(["fetch", "origin", "--prune"]).stdout)
    remote_branch = subprocess.run(
        ["git", "ls-remote", "--heads", "origin", branch],
        cwd=repo,
        check=True,
        text=True,
        capture_output=True,
    ).stdout.strip()
    if remote_branch:
        pull = git(["pull", "--ff-only", "origin", branch])
        if pull.stdout.strip():
            print(pull.stdout)
    else:
        print(f"remote branch {branch} does not exist yet; push will create it")
    push = git(["push", "-u", "origin", branch])
    print(push.stdout)
    print(git(["status", "--short", "--branch"]).stdout)
finally:
    git(["config", "--local", "--unset", extraheader_key], check=False)
    token = ""
    basic_auth = ""

repo: /content/v2e
branch: research/apr03


CalledProcessError: Command '['git', 'fetch', 'origin', '--prune']' returned non-zero exit status 128.

In [36]:
# Optuna has saturated the hyperparameter space (6 trials, best 0.5126 << 0.5865).
# Try architectural modification: increase UNet depth (more layers).

import os
import subprocess
from pathlib import Path

_repo = globals().get("ROOT", Path("/content/v2e"))
_train_file = globals().get("TRAIN_FILE", _repo / "research/v2e_imu/train.py")

os.chdir(_repo)

# Read current training config
_text = _train_file.read_text(encoding="utf-8")
print("Current model config:")
for _line in _text.splitlines():
    if any(x in _line for x in ["BASE_CHANNELS", "LAYERS", "DEPTH"]):
        print(f"  {_line.strip()[:80]}")

# Test: increase BASE_CHANNELS from 32 to 48 (50% more capacity)
_new_text = _text.replace("BASE_CHANNELS = 32", "BASE_CHANNELS = 48")

if _new_text != _text:
    _train_file.write_text(_new_text, encoding="utf-8")
    print("\nUpdated BASE_CHANNELS: 32 → 48 (50% capacity increase)")

    # Commit and run
    subprocess.run(["git", "add", str(_train_file.relative_to(_repo))], cwd=_repo, check=True)
    subprocess.run(
        ["git", "commit", "-m", "experiment: increase UNet capacity (BASE_CHANNELS 32→48)"],
        cwd=_repo,
        check=True,
    )
    _sha = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=_repo,
        check=True,
        text=True,
        capture_output=True,
    ).stdout.strip()
    print(f"Committed: {_sha}")
else:
    print("No changes needed")

Current model config:
  which would otherwise happen if the autoresearch agent tries BASE_CHANNELS not
  BASE_CHANNELS = 32
  FNO_LAYERS = 4  # Number of FNO+FiLM blocks
  fno_layers=FNO_LAYERS,
  config = ModelConfig(base_channels=BASE_CHANNELS, imu_hidden_dim=IMU_HIDDEN_DIM)
  print(f"base_channels: {BASE_CHANNELS}")
  "base_channels": BASE_CHANNELS,
  "fno_layers": FNO_LAYERS,

Updated BASE_CHANNELS: 32 → 48 (50% capacity increase)
Committed: 6f5c106


In [37]:
import os
import subprocess
from pathlib import Path

_repo = globals().get("ROOT", Path("/content/v2e"))
_run_log_file = globals().get("RUN_LOG", _repo / "research/v2e_imu/run_log.txt")

# Import helper functions from Cell 5 (setup)
_run_train = globals().get("run_train_with_timeout")
_parse = globals().get("parse_run_metrics")
_append = globals().get("append_result")

os.chdir(_repo)

print("Running training with BASE_CHANNELS=48...")
_ok, _ = _run_train()
_log = _run_log_file.read_text(encoding="utf-8", errors="replace") if _run_log_file.exists() else ""
_val_ap, _peak_vram = _parse(_log)

print(f"Result: ok={_ok}, val_ap={_val_ap}, peak_vram={_peak_vram}")

if _ok and _val_ap is not None and _peak_vram is not None:
    _sha = "6f5c106"
    _best = 0.586500
    _status = "keep" if _val_ap > _best else "discard"
    _desc = "increased UNet BASE_CHANNELS 32→48 (+50% capacity)"

    _append(_sha, _val_ap, _peak_vram, _status, _desc)
    print(f"Result logged: {_status} val_ap={_val_ap:.6f}")

    if _status == "discard":
        print("Reverting...")
        subprocess.run(["git", "reset", "--hard", "HEAD~1"], cwd=_repo, check=True)
else:
    print("Training failed or incomplete metrics")

Running training with BASE_CHANNELS=48...
Result: ok=True, val_ap=0.4635, peak_vram=0.0
Result logged: discard val_ap=0.463500
Reverting...


In [38]:
# Targeted architecture probe: larger IMU encoder with best-known training schedule.
import os
import re
import subprocess
from pathlib import Path

_repo = globals().get("ROOT", Path("/content/v2e"))
_train_file = globals().get("TRAIN_FILE", _repo / "research/v2e_imu/train.py")

os.chdir(_repo)

_text = _train_file.read_text(encoding="utf-8")

# Lock known-strong base settings from prior best run, then change one architecture knob.
_replacements = {
    "TOTAL_BATCH_SIZE": "4",
    "DEVICE_BATCH_SIZE": "4",
    "LEARNING_RATE": "1e-3",
    "WEIGHT_DECAY": "0.0",
    "WARMUP_RATIO": "0.1",
    "WARMDOWN_RATIO": "0.3",
    "FINAL_LR_FRAC": "0.01",
    "MODEL_TYPE": '"unet"',
    "BASE_CHANNELS": "32",
    "IMU_HIDDEN_DIM": "192",  # architecture change
}

_new_text = _text
for _k, _v in _replacements.items():
    _new_text = re.sub(rf"^{_k}\s*=\s*.*$", f"{_k} = {_v}", _new_text, count=1, flags=re.MULTILINE)

if _new_text == _text:
    print("No changes detected in train.py")
else:
    _train_file.write_text(_new_text, encoding="utf-8")
    print("Applied config updates:")
    for _k, _v in _replacements.items():
        print(f"  {_k} = {_v}")

    subprocess.run(["git", "add", str(_train_file.relative_to(_repo))], cwd=_repo, check=True)
    _rc = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=_repo, check=False).returncode
    if _rc == 0:
        print("Nothing staged; skipping commit")
    else:
        subprocess.run(
            [
                "git",
                "commit",
                "-m",
                "experiment: IMU_HIDDEN_DIM 128->192 with best known scheduler",
            ],
            cwd=_repo,
            check=True,
        )
        print(
            "Committed:",
            subprocess.run(
                ["git", "rev-parse", "--short", "HEAD"],
                cwd=_repo,
                text=True,
                capture_output=True,
                check=True,
            ).stdout.strip(),
        )

Applied config updates:
  TOTAL_BATCH_SIZE = 4
  DEVICE_BATCH_SIZE = 4
  LEARNING_RATE = 1e-3
  WEIGHT_DECAY = 0.0
  WARMUP_RATIO = 0.1
  WARMDOWN_RATIO = 0.3
  FINAL_LR_FRAC = 0.01
  MODEL_TYPE = "unet"
  BASE_CHANNELS = 32
  IMU_HIDDEN_DIM = 192
Committed: d775f0e


In [ ]:
# Train/evaluate latest experiment commit and keep/discard against current best.
import os
import subprocess
from pathlib import Path

_repo = globals().get("ROOT", Path("/content/v2e"))
_run_log_file = globals().get("RUN_LOG", _repo / "research/v2e_imu/run.log")

_run_train = globals().get("run_train_with_timeout")
_parse = globals().get("parse_run_metrics")
_append = globals().get("append_result")
_short_commit = globals().get("short_commit")
_exp_desc = globals().get("EXP_DESC", "manual architecture experiment")

if not all([_run_train, _parse, _append, _short_commit]):
    raise RuntimeError("Missing setup helpers. Run setup cell first.")

os.chdir(_repo)
_cur_sha = _short_commit()
print(f"Running training for commit {_cur_sha}...")
print(f"Description: {_exp_desc}")
_ok, _ = _run_train()
_log = _run_log_file.read_text(encoding="utf-8", errors="replace") if _run_log_file.exists() else ""
_val_ap, _peak_vram = _parse(_log)
print(f"Result: ok={_ok}, val_ap={_val_ap}, peak_vram={_peak_vram}")

if _ok and _val_ap is not None and _peak_vram is not None:
    _best = 0.586500
    _status = "keep" if _val_ap > _best else "discard"
    _append(_cur_sha, _val_ap, _peak_vram, _status, _exp_desc)
    print(f"Logged: {_status} val_ap={_val_ap:.6f}")
    if _status == "discard":
        subprocess.run(["git", "reset", "--hard", "HEAD~1"], cwd=_repo, check=True)
        print("Reverted discarded commit")
else:
    print("Training failed or metrics missing; logging crash")
    _append(_cur_sha, 0.0, 0.0, "crash", f"{_exp_desc} failed")
    subprocess.run(["git", "reset", "--hard", "HEAD~1"], cwd=_repo, check=True)

Running training for commit 7b5bf5b...
Description: smaller UNet width: BASE_CHANNELS 32->24 (best scheduler fixed)


In [40]:
# Targeted architecture probe: deeper IMU temporal model (2 -> 3 LSTM layers).
import os
import re
import subprocess
from pathlib import Path

_repo = globals().get("ROOT", Path("/content/v2e"))
_train_file = globals().get("TRAIN_FILE", _repo / "research/v2e_imu/train.py")

os.chdir(_repo)

_text = _train_file.read_text(encoding="utf-8")
_new_text = _text

# Lock known-strong scalar settings.
for _k, _v in {
    "TOTAL_BATCH_SIZE": "4",
    "DEVICE_BATCH_SIZE": "4",
    "LEARNING_RATE": "1e-3",
    "WEIGHT_DECAY": "0.0",
    "WARMUP_RATIO": "0.1",
    "WARMDOWN_RATIO": "0.3",
    "FINAL_LR_FRAC": "0.01",
    "MODEL_TYPE": '"unet"',
    "BASE_CHANNELS": "32",
    "IMU_HIDDEN_DIM": "128",
}.items():
    _new_text = re.sub(rf"^{_k}\s*=\s*.*$", f"{_k} = {_v}", _new_text, count=1, flags=re.MULTILINE)

# Architecture change: IMUEncoder num_layers default from 2 to 3.
_new_text = _new_text.replace(
    "def __init__(self, input_dim: int = 6, hidden_dim: int = 128, num_layers: int = 2) -> None:",
    "def __init__(self, input_dim: int = 6, hidden_dim: int = 128, num_layers: int = 3) -> None:",
)

if _new_text == _text:
    print("No changes detected in train.py")
else:
    _train_file.write_text(_new_text, encoding="utf-8")
    EXP_DESC = "deeper IMU encoder: bidirectional LSTM layers 2->3 (best scheduler fixed)"
    print("Prepared:", EXP_DESC)

    subprocess.run(["git", "add", str(_train_file.relative_to(_repo))], cwd=_repo, check=True)
    _rc = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=_repo, check=False).returncode
    if _rc == 0:
        print("Nothing staged; skipping commit")
    else:
        subprocess.run(
            ["git", "commit", "-m", "experiment: IMU LSTM depth 2->3"],
            cwd=_repo,
            check=True,
        )
        print(
            "Committed:",
            subprocess.run(
                ["git", "rev-parse", "--short", "HEAD"],
                cwd=_repo,
                text=True,
                capture_output=True,
                check=True,
            ).stdout.strip(),
        )

Prepared: deeper IMU encoder: bidirectional LSTM layers 2->3 (best scheduler fixed)
Committed: 5d6a492


In [42]:
# Targeted architecture probe: smaller UNet width (32 -> 24) for regularization.
import os
import re
import subprocess
from pathlib import Path

_repo = globals().get("ROOT", Path("/content/v2e"))
_train_file = globals().get("TRAIN_FILE", _repo / "research/v2e_imu/train.py")

os.chdir(_repo)

_text = _train_file.read_text(encoding="utf-8")
_new_text = _text

# Restore IMU encoder depth to baseline 2 if needed.
_new_text = _new_text.replace(
    "def __init__(self, input_dim: int = 6, hidden_dim: int = 128, num_layers: int = 3) -> None:",
    "def __init__(self, input_dim: int = 6, hidden_dim: int = 128, num_layers: int = 2) -> None:",
)

for _k, _v in {
    "TOTAL_BATCH_SIZE": "4",
    "DEVICE_BATCH_SIZE": "4",
    "LEARNING_RATE": "1e-3",
    "WEIGHT_DECAY": "0.0",
    "WARMUP_RATIO": "0.1",
    "WARMDOWN_RATIO": "0.3",
    "FINAL_LR_FRAC": "0.01",
    "MODEL_TYPE": '"unet"',
    "BASE_CHANNELS": "24",  # architecture change
    "IMU_HIDDEN_DIM": "128",
}.items():
    _new_text = re.sub(rf"^{_k}\s*=\s*.*$", f"{_k} = {_v}", _new_text, count=1, flags=re.MULTILINE)

if _new_text == _text:
    print("No changes detected in train.py")
else:
    _train_file.write_text(_new_text, encoding="utf-8")
    EXP_DESC = "smaller UNet width: BASE_CHANNELS 32->24 (best scheduler fixed)"
    print("Prepared:", EXP_DESC)

    subprocess.run(["git", "add", str(_train_file.relative_to(_repo))], cwd=_repo, check=True)
    _rc = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=_repo, check=False).returncode
    if _rc == 0:
        print("Nothing staged; skipping commit")
    else:
        subprocess.run(
            ["git", "commit", "-m", "experiment: UNet width 32->24"],
            cwd=_repo,
            check=True,
        )
        print(
            "Committed:",
            subprocess.run(
                ["git", "rev-parse", "--short", "HEAD"],
                cwd=_repo,
                text=True,
                capture_output=True,
                check=True,
            ).stdout.strip(),
        )

Prepared: smaller UNet width: BASE_CHANNELS 32->24 (best scheduler fixed)
Committed: 7b5bf5b
